Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [10]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic

In [11]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    map = rng.random(size=(size, 2))
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()

In [ ]:
problem = create_problem(10, density=0.15, noise_level=10, negative_values=False)
NUM = 10

In [13]:
problem

array([[    0.,    inf, 10573.,    inf,  8430.,    inf,    inf,   831.,
         1977.,    inf],
       [   inf,     0.,    inf,    inf,    inf,  2434.,    inf,    inf,
           inf,  6771.],
       [   inf,    inf,     0.,    inf,    inf,    inf,    inf,    inf,
           inf,  2208.],
       [   inf,    inf,  8563.,     0.,  7768.,    inf,    inf,    inf,
           inf,    inf],
       [ 7330.,    inf,    inf,  8367.,     0.,    inf,    inf,    inf,
           inf,    inf],
       [   inf,    inf,    inf,    inf,    inf,     0.,    inf,    inf,
           inf,    inf],
       [   inf,    inf,    inf,    inf,    inf,    inf,     0.,  5247.,
           inf,    inf],
       [   inf,    inf,  5287.,    inf,    inf,    inf,    inf,     0.,
           inf,    inf],
       [   inf,    inf,    inf, 10443.,  8363.,    inf,  5258.,    inf,
            0.,    inf],
       [   inf,    inf,    inf,    inf,    inf,    inf,  7850.,  7752.,
           inf,     0.]])

In [14]:
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [15]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        path = nx.shortest_path(G, s, d, weight='weight')
        cost  = nx.path_weight(G, path, weight='weight')
    except nx.NetworkXNoPath:
        path = None
        cost = np.inf
    ic(s, d, path, cost)
None

ic| s: 0, d: 1, path: None, cost: inf
ic| s: 0, d: 2, path: [0, 7, 2], cost: 6118.0
ic| s: 0, d: 3, path: [0, 8, 3], cost: 12420.0
ic| s: 0, d: 4, path: [0, 4], cost: 8430.0
ic| s: 0, d: 5, path: None, cost: inf
ic| s: 0, d: 6, path: [0, 8, 6], cost: 7235.0
ic| s: 0, d: 7, path: [0, 7], cost: 831.0
ic| s: 0, d: 8, path: [0, 8], cost: 1977.0
ic| s: 0, d: 9, path: [0, 7, 2, 9], cost: 8326.0
ic| s: 1, d: 2, path: [1, 9, 7, 2], cost: 19810.0
ic| s: 1, d: 3, path: None, cost: inf
ic| s: 1, d: 4, path: None, cost: inf
ic| s: 1, d: 5, path: [1, 5], cost: 2434.0
ic| s: 1, d: 6, path: [1, 9, 6], cost: 14621.0
ic| s: 1, d: 7, path: [1, 9, 7], cost: 14523.0
ic| s: 1, d: 8, path: None, cost: inf
ic| s: 1, d: 9, path: [1, 9], cost: 6771.0
ic| s: 2, d: 3, path: None, cost: inf
ic| s: 2, d: 4, path: None, cost: inf
ic| s: 2, d: 5, path: None, cost: inf
ic| s: 2, d: 6, path: [2, 9, 6], cost: 10058.0
ic| s: 2, d: 7, path: [2, 9, 7], cost: 9960.0
ic| s: 2, d: 8, path: None, cost: inf
ic| s: 2, d: 9, pat

In [17]:
"""
import logging
from queue import PriorityQueue
from icecream import Callable


def search(
    initial_state: State,
    parent_state: dict,
    state_cost: dict,
    priority_function: Callable,
    unit_cost: Callable,
):
    frontier = PriorityQueue()
    parent_state.clear()
    state_cost.clear()

    state = initial_state
    parent_state[state] = None
    state_cost[state] = 0

    while state is not None and not goal_test(state):
        for a in possible_actions(state):
            new_state = result(state, a)
            cost = unit_cost(a)
            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                state_cost[new_state] = state_cost[state] + cost
                frontier.push(new_state, p=priority_function(new_state))
                logging.debug(f"Added new node to frontier (cost={state_cost[new_state]})")
            elif new_state in frontier and state_cost[new_state] > state_cost[state] + cost:
                old_cost = state_cost[new_state]
                parent_state[new_state] = state
                state_cost[new_state] = state_cost[state] + cost
                logging.debug(
                    f"Updated node cost in frontier: {old_cost} -> {state_cost[new_state]}"
                )
        if frontier:
            state = frontier.pop()
        else:
            state = None

    path = list()
    s = state
    while s:
        path.append(s.copy_data())
        s = parent_state[s]

    logging.info(f"Found a solution in {len(path):,} steps; visited {len(state_cost):,} states")
    return list(reversed(path))
    """


'\nimport logging\nfrom queue import PriorityQueue\nfrom icecream import Callable\n\n\ndef search(\n    initial_state: State,\n    parent_state: dict,\n    state_cost: dict,\n    priority_function: Callable,\n    unit_cost: Callable,\n):\n    frontier = PriorityQueue()\n    parent_state.clear()\n    state_cost.clear()\n\n    state = initial_state\n    parent_state[state] = None\n    state_cost[state] = 0\n\n    while state is not None and not goal_test(state):\n        for a in possible_actions(state):\n            new_state = result(state, a)\n            cost = unit_cost(a)\n            if new_state not in state_cost and new_state not in frontier:\n                parent_state[new_state] = state\n                state_cost[new_state] = state_cost[state] + cost\n                frontier.push(new_state, p=priority_function(new_state))\n                logging.debug(f"Added new node to frontier (cost={state_cost[new_state]})")\n            elif new_state in frontier and state_cost[new_s

In [ ]:
node_attributes = nx.get_node_attributes(G, 'City index') 
print("Node Attributes:", node_attributes)

# Retrieve and print edge attributes
edge_weights = nx.get_edge_attributes(G, 'weight')  # Get 'weight' attribute for all edges
print("Edge Attributes (Weight):", edge_weights)

# Draw the graph with node and edge attributes
pos = nx.spring_layout(G)  # Layout for node positions
node_labels = nx.get_node_attributes(G, 'label')  # Get node labels for visualization
edge_labels = nx.get_edge_attributes(G, 'weight')  # Get edge weights for visualization


Node Attributes (Age): {}
Edge Attributes (Weight): {(0, 2): 10573.0, (0, 4): 8430.0, (0, 7): 831.0, (0, 8): 1977.0, (1, 5): 2434.0, (1, 9): 6771.0, (2, 9): 2208.0, (3, 2): 8563.0, (3, 4): 7768.0, (4, 0): 7330.0, (4, 3): 8367.0, (6, 7): 5247.0, (7, 2): 5287.0, (8, 3): 10443.0, (8, 4): 8363.0, (8, 6): 5258.0, (9, 6): 7850.0, (9, 7): 7752.0}


In [ ]:
import logging
from queue import PriorityQueue

def dijkstra_search(
    graph,
    start,
    goal,
    parent_state,
    state_cost,
):
    """
    Dijkstra using professor's standard search() interface.
    priority = current state_cost
    unit_cost(a) returns the edge weight
    """

    # --------------------------
    # Required template functions
    # --------------------------
    
    def goal_test(state):
        return state == goal

    def possible_actions(state):
        return graph[state]  # NetworkX => adjacency dict of neighbors

    def result(state, action):
        return action  # Action is simply "which neighbor"

    def unit_cost(action):
        # graph[state][action] is dict -> {'weight': cost}
        return graph[state][action].get("weight", 1)

    def priority_function(state):
        return state_cost[state]  # Dijkstra priority = g(n)

    # --------------------------
    # Frontier with cost support
    # --------------------------
    class FrontierPQ:
        def __init__(self):
            self.pq = PriorityQueue()
            self.entry_map = {}  # state -> priority

        def push(self, state, p):
            self.entry_map[state] = p
            self.pq.put((p, state))

        def pop(self):
            # pop until we find a non-stale entry
            while not self.pq.empty():
                p, s = self.pq.get()
                if s in self.entry_map and self.entry_map[s] == p:
                    del self.entry_map[s]  # remove now
                    return s
            return None

        def __contains__(self, state):
            return state in self.entry_map

        def __bool__(self):
            return len(self.entry_map) > 0

    # --------------------------
    # Initialize
    # --------------------------
    frontier = FrontierPQ()
    parent_state.clear()
    state_cost.clear()

    state = start
    parent_state[state] = None
    state_cost[state] = 0
    frontier.push(state, p=priority_function(state))

    # --------------------------
    # Main loop
    # --------------------------
    while state is not None and not goal_test(state):
        for a in possible_actions(state):
            new_state = result(state, a)
            cost = unit_cost(a)
            new_g = state_cost[state] + cost

            # Case 1: not visited and not in frontier
            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                state_cost[new_state] = new_g
                frontier.push(new_state, p=priority_function(new_state))

            # Case 2: in frontier with higher cost → decrease-key
            elif new_state in frontier and new_g < state_cost[new_state]:
                old_cost = state_cost[new_state]
                parent_state[new_state] = state
                state_cost[new_state] = new_g
                frontier.push(new_state, p=priority_function(new_state))  # update

        # Advance search
        if frontier:
            state = frontier.pop()
        else:
            state = None

    # --------------------------
    # Reconstruct path
    # --------------------------
    if state is None:
        return None, float("inf")

    path = []
    s = state
    while s is not None:
        path.append(s)
        s = parent_state[s]

    path.reverse()
    return path, state_cost[state]


In [ ]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        nx_path = nx.shortest_path(G, s, d, weight='weight')
        nx_cost  = nx.path_weight(G, nx_path, weight='weight')
    except nx.NetworkXNoPath:
        nx_path = None
        nx_cost = np.inf
        # Our Dijkstra implementation
    parent_state = {}
    state_cost = {}
    
    def dummy_priority(state):
        return state_cost.get(state, float('inf'))
    
    def dummy_unit_cost(state, action):
        return 1  # Not used directly in our implementation
    
    dijkstra_path, dijkstra_cost = dijkstra_search(
        G, s, d, parent_state, state_cost, dummy_priority, dummy_unit_cost
    )
    
    print(f"From {s} to {d}:")
    print(f"  NetworkX: path={nx_path}, cost={nx_cost}")
    print(f"  Dijkstra: path={dijkstra_path}, cost={dijkstra_cost}")
    print(f"  Match: {nx_cost == dijkstra_cost and nx_path == dijkstra_path}")
    print()
    
    ic(s, d, nx_path, nx_cost, dijkstra_path, dijkstra_cost)